<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 5: Eager and Lazy APIs

In [ ]:
%%bash

pip install --upgrade polars

In [ ]:
import polars as pl
pl.__version__

## Eager API: DataFrame

### Downloading Multiple Parquet Files

You can use `curl`'s brace expansion feature to download multiple sequential months in one command.
- `{01,02,03}` specifies the months to fetch.
- `-o` uses matching output paths where `#1` represents the pattern matched inside the braces (the month).

In [ ]:
%%bash

curl -L --create-dirs \
  -o "sample_data/taxi/yellow_tripdata_2022-#1.parquet" \
  "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data/taxi/yellow_tripdata_2022-{01,02,03,04,05,06,07,08,09,10,11,12}.parquet"

In [ ]:
!ls -lh sample_data/taxi/

In [ ]:
%%time
trips = pl.read_parquet("sample_data/taxi/yellow_tripdata_2022-*.parquet")

# Fix: Sum only the columns we need using agg() instead of calling sum() on the entire group
sum_per_vendor = trips.group_by("VendorID").agg(
    pl.col("total_amount").sum(),
    pl.col("trip_distance").sum()
)

income_per_distance_per_vendor = sum_per_vendor.select(
    "VendorID",
    income_per_distance=pl.col("total_amount") / pl.col("trip_distance"),
)

top_three = income_per_distance_per_vendor.sort(
    by="income_per_distance", descending=True
).head(3)

top_three

## Lazy API: LazyFrame

In [ ]:
# This raises a SchemaError:
names_lf = pl.LazyFrame({"name": ["Alice", "Bob", "Charlie"], "age": [25, 30, 35]})

erroneous_query = names_lf.with_columns(
     sliced_age=pl.col("age").str.slice(1, 3)
)

result_df = erroneous_query.collect()

## Performance Differences

In [ ]:
%%time
trips = pl.scan_parquet("sample_data/taxi/yellow_tripdata_2022-*.parquet")

# Fix: Sum only the columns we need using agg() instead of calling sum() on the entire group
sum_per_vendor = trips.group_by("VendorID").agg(
    pl.col("total_amount").sum(),
    pl.col("trip_distance").sum()
)

income_per_distance_per_vendor = sum_per_vendor.select(
    "VendorID",
    income_per_distance=pl.col("total_amount") / pl.col("trip_distance"),
)

top_three = income_per_distance_per_vendor.sort(
    by="income_per_distance", descending=True
).head(3)

top_three.collect()

In [ ]:
lf = pl.LazyFrame({"col1": [1, 2, 3], "col2": [4, 5, 6]})

# ... Some heavy computation ...

print(lf.collect())

print(lf.with_columns(pl.col("col1") + 1).collect())

## Tips and Tricks

### Going from LazyFrame to DataFrame and Vice Versa

### Joining a DataFrame with a LazyFrame

In [ ]:
# This raises a TypeError:
big_sales_data = pl.LazyFrame(
    {"sale_id": [101, 102, 103], "amount": [250, 150, 300]}
)

sales_metadata = pl.DataFrame(
    {"sale_id": [101, 102, 103], "category": ["A", "B", "A"]}
)

big_sales_data.join(sales_metadata, on="sale_id").collect()

In [ ]:
big_sales_data = pl.LazyFrame(
    {"sale_id": [101, 102, 103], "amount": [250, 150, 300]}
)

sales_metadata = pl.DataFrame(
    {"sale_id": [101, 102, 103], "category": ["A", "B", "A"]}
)

big_sales_data.join(sales_metadata.lazy(), on="sale_id").collect()

### Caching Intermittent Results

In [ ]:
lf = pl.LazyFrame({"col1": [1, 2, 3], "col2": [4, 5, 6]})

# ... Some heavy computation ...

lf = lf.collect().lazy()
print(lf.collect())

print(lf.with_columns(pl.col("col1") + 1).collect())